*Friday, 14 November 2025*

In [1]:

# PYTHON LIBRARIES
# CGS Module:
import sys
import os
sys.path.append(os.path.abspath(r"..\cgsMPDT1"))
# from crcsspace import CRCSSpace
from cgs import CGSMPDT1

# General Modules:
import pandas as pd
import numpy as np
from math import floor
import time
from IPython.display import Latex, HTML
from dash import html

title = "The Computational Notebook of CGS Optimization for the BPPT Concreting Works"
title_html = f"""
<link href="https://fonts.googleapis.com/css2?family=Poppins:wght@400;600&display=swap" rel="stylesheet">
<div style="
    text-align: center;
    font-weight: bold;
    font-family: 'Poppins', sans-serif;
    color: steelblue;
    font-size: 40px;
    ">
    {title}
</div>
"""
display(HTML(title_html))

# 1. Introduction

This notebook contains the computational process of optimizing the concreting works in the BPPT construction project, which is the main topic in our research manuscript entitled ***A Computational Implementation of Combinatorial Grid Sequencing Theory for Optimizing Concreting Cycle with Variations in Formwork and Shoring Transfers***. We implement the theory of Combinatorial Grid Sequencing (CGS) computationally to model the concreting work sequence and to build an objective function of the optimization.

Our aim for the optimization is finding a concreting configuration consisting of the number of formworks and shoring and their renting duration such that the concreting work is finished within a maximum time simliar to that of the manually optimized configuration with the least cost. In our manual simulation, we found an equivalent CGS space (configuration) with the heuristically optimized result presented as follows:
- CGS space: $(\{p_1, p_2, p_3\}, \{s_1\}, (3, 5), (1, 1, 3, 3, 3))$
- Total concreting duration (days): $\mathfrak{T}(0, \varepsilon, \dotsc, \varepsilon) = 46$
- Material cost: 450,591,276 (96.94%)
- Formwork renting cost: 99,105,230.61 (38%)
- Shoring renting cost: 66,070,153.85 (40.29%)
- Manpower cost: 649,600,339 (96.39%)
- Total cost: 610,133,352.85 (70%)

The percentage in the data above is compared with the actual project execution.

Therefore, our objective is **finding a CGS space with the least cost which has a maximum concreting duration less than or equal to 46 days**.

# 2. Modelling

## 2.1. Preliminary Information

The preliminary information required for modelling the BPPT building concreting sequence using the theory of CGS includes the *number of floors*, the number of *horizontal division for construction process*, predefined time information such as rebars and concreting preparations and internal time process. The detail information is presented as follows.

|Parameter|Value|Unit|
|---------|-----|----|
|Num. Floor|5|days|
|Num. Hor. Div.|4|   |
|Reb. Preparation|1|days|
|Conc. Preparation|1|days|
|Installation Duration|3|days|
|Conc. Age|3, 7, 10, 14, 21, 28|days|

In addition, the manpower cost is estimated to be linear with respect to total concreting duration and follows the following formula:
$$
\mathrm{manpower}(t) = 14121746.5 t
$$
with the unit in IDR.

## 2.2. CGS Spaces Representation

### Filtering Subfamily of CGS Spaces

Note that our optimization formally aims to find a CGS space that minimizes an objective function which considers total concreting time and cost. We do not need to find one from the entire family of CGS spaces, but from a subfamily filtered based on our preliminary information. The detailed constraints are presented as follows.
- This problem is properly modelled with some CGS space equipped with CGS-MPD-T1 (see the manuscript of CGS theory).
- The shape of CGS spaces are $\mathbf{n} = (3, 5)$, since there are 4 total horizontal division and 5 floors.
- The CGS dynamic parameter for this specific problem is some $\mathbf{d} \in \mathbb{N}^5$.
- The first 3 entries of the dynamic parameter $\mathbf{d} \in \mathbb{N}^5$ is equal to $(1, 1, 3)$, given that the rebar concrete installing preparations are both 1 day and the internal process is 3 days.
- The last 2 entries of the dynamic parameter $\mathbf{d} \in \mathbb{N}^5$ is within the set
$$
D_{\mathrm{conc}} := \{3, 7, 10, 14, 21, 28\} \,,
$$
which is the set of concrete ages.

Follows from the manuscript of the CGS space, we denote the family of all CGS spaces by $\mathcal{C}$. The filtered subfamily required in our optimization is given by $\tilde{\mathcal{C}} \subset \mathcal{C}$ such that
$$
\tilde{\mathcal{C}}
= \{ (A_p, A_s, \mathbf{n}, \mathbf{d}) \in \mathcal{C}
\mid \mathbf{n} = (3, 5)
\land \mathbf{d} \in \mathbb{N}^5
\land (d_1, d_2, d_3) = (1, 1, 3)
\land d_4, d_5 \in D_{\mathrm{conc}}
\} \,.
$$
For convenience, we construct a set $X$ which can represent $\tilde{\mathcal{C}}$ given by
$$
X := \{ (|A_p|, |A_s|, n_1, n_2, d_1, \dotsc, d_5) \in \mathbb{Z}^9
\mid (A_p, A_s, (n_1, n_2), (d_1, \dotsc, d_5)) \in \tilde{\mathcal{C}} \}
\,.
$$
Note that $X$ and $\tilde{\mathcal{C}}$ are bijective. Let $\beta: X \to \tilde{\mathcal{C}}$ be the bijection, which is defined by
$$
\beta(|A_p|, |A_s|, n_1, n_2, d_1, \dotsc, d_5) := (A_p, A_s, (n_1, n_2), (d_1, \dotsc, d_5))
$$
for any $(|A_p|, |A_s|, n_1, n_2, d_1, \dotsc, d_5) \in X$. In other words, $\beta$ is a two-way bridge between $X$ and $\tilde{\mathcal{C}}$. For convenience, we may only denote an element of $X$ by $\mathbf{x} \in X$ instead of a detailed tuple notation.

In addition, all CGS spaces in $\tilde{\mathcal{C}}$ has the same CGS-SS (CGS State Space), which is $\mathbb{Z}_{\max}^{20}$(the number 20 is from $(3 + 1) 5 = 20$).

### Objective Function

We will minimize the cost of elements of $X$ which have the total concreting time up to 45 days. The cost comprises of the renting costs of formwork and shoring, and the cost of concreting material according to its immediate strength.


**Renting Cost**

Let $\mathrm{proj}_k: X \to \mathbb{N}$ be the canonical projection on $X$, i.e.,
$$
\mathrm{proj}_k(\mathbf{x})
= \mathrm{proj}_k(x_1, \dotsc, x_9)
:= x_k
$$
for any $k \in \{1, \dotsc, 9\}$. The corresponding number of formwork and shoring in the primary and secondary area are given by $\mathrm{proj}_1$ and $\mathrm{proj}_2$ (cardinality of the PAC and SAC) respectively. The renting cost function is given by a map $\Psi_{\mathrm{rent}}: X \to [0, \infty)$ which is defined by
$$
\Psi_{\mathrm{rent}}(\mathbf{x})
:= \Big[
\mathfrak{T}_p^{\beta(\mathbf{x})}(\mathbf{u}_0)
\cdot \mathrm{proj}_1(\mathbf{x}) \cdot \left( \mathrm{rent}_p^{form} + \mathrm{rent}_p^{shor} \right)
\Big]
+ \Big[
\mathfrak{T}_s^{\beta(\mathbf{x})}(\mathbf{u}_0)
\cdot \mathrm{proj}_2(\mathbf{x}) \cdot \left( \mathrm{rent}_s^{form} + \mathrm{rent}_s^{shor} \right)
\Big]
$$
for every $\mathbf{x} \in X$, where $\mathfrak{T}_p^{\beta(\mathbf{x})}(\mathbf{u}_0)$ is the renting duration of equipments in the primary area, $\mathfrak{T}_p^{\beta(\mathbf{x})}(\mathbf{u}_0)$ is the renting duration of equipments in the secondary area, $\mathbf{u}_0 := (0, \varepsilon, \dotsc, \varepsilon) \in \mathbb{Z}_{\max}^{20}$ is the starting state (schedule), and $\mathrm{rent}_p^{form}, \mathrm{rent}_p^{shor}, \mathrm{rent}_s^{form}, \mathrm{rent}_s^{shor} \in [0, \infty)$ are price density of formwork and shoring respectively.


**Material Cost**

The concrete age of immediate strength in use is given by $\mathrm{proj}_8$. And the concrete material cost function is given by a map $\Psi_{\mathrm{conc}}: X \to [0, \infty)$ defined by
$$
\Psi_{\mathrm{conc}}(\mathbf{x})
:= (\Gamma \circ \mathrm{proj}_8)(\mathbf{x})
$$
for every $\mathbf{x} \in X$, where the map $\Gamma: \mathbb{N} \to [0, \infty)$ describes the cost of the concrete material given the age of immediate strength.


**Manpower Cost**

The manpower cost follows the formula presented in the introduction. In terms of the objective function, it is given by a map $\Psi_{\mathrm{mp}}: X \to [0, \infty)$ defined by
$$
\Psi_{\mathrm{mp}} := \mathrm{manpower} \circ \mathfrak{T}
$$
where $\mathfrak{T}: \mathbb{Z}_{\max}^{|\mathcal{E}|} \to \mathbb{Z}_{\max}$ is the CGS-F representing the total concreting duration.


The total objective function is given by a map $\Psi: X \to [0, \infty)$ defined by
$$
\Psi(x) := \Psi_{\mathrm{rent}}(\mathbf{x}) + \Psi_{\mathrm{conc}}(\mathbf{x}) + \Psi_{\mathrm{mp}}(\mathbf{x})
\,,
$$
for every $\mathbf{x} \in X$.

By the imposed condition that the maximum total concreting duration is 45 days, the feasible set is refined into a set
$$
X_{46} := \left\{ \mathbf{x} \in X \;\middle|\; \mathfrak{T}^{\beta(\mathbf{x})}(\mathbf{u}_0) \leq 46 \right\} \,.
$$
An optimal solution is a point $\mathbf{x}^\ast \in X_{46}$ such that
$$
\mathbf{x}^\ast = \underset{\mathbf{x} \in X_{46}}{\arg\min}\; \Psi(\mathbf{x}) \,.
$$

## 2.3. Python Script of the Objective Function

It is worth noting that the concrete material cost is constant. We implement $\Psi: X \to [0, \infty)$ in Python as follows.

In [2]:
# OBJECTIVE FUNCTION

def beta(x):
    lenPAC, lenSAC = x[0], x[1]
    n = (x[2], x[3])
    d = tuple(x[4:])
    return (lenPAC, lenSAC, n, d)

def Psi(
        x:list,
        rent_pFORM= 469442.31,
        rent_pSHOR= 312961.54,
        rent_sFORM= 928471.15,
        rent_sSHOR= 618980.77
        ):
    cgs_sp = beta(x)
    proj1, proj2, _, _ = cgs_sp
    proj8 = x[7]
    cgs = CGSMPDT1(*cgs_sp)
    frakT, frakT_p, frakT_s = cgs.cgs_allF()

    Gamma = lambda u: 450591276
    
    rent = frakT_p *proj1 *(rent_pFORM + rent_pSHOR) \
        + frakT_s *proj2 *(rent_sFORM + rent_sSHOR)
    conc = Gamma(proj8)
    mp = 14121746.5 *frakT

    return rent + conc + mp

## 2.4. Optimization Method

We are opting the exhaustive search (ES), a.k.a. brute force, method. A solution obtained from ES is a global optimum, and this is the major advantage of employing ES. However, ES is computationally very expensive, and when the feasible set is very large or even infinite, ES is not a reasonable choice. However, our feasible set is likely relatively small. Therefore, we will provide a justification in using ES by evaluating the cardinality of feasible set and the empirical computational time when evaluating each point in the feasible set.

### Cardinality of Feasible Set

Follows from the CGS theory, the cardinality of the PAC cannot exceed $n_1 \cdot n_2 = 3 \cdot 5 = 15$, and the cardinality of SAC cannot exceed $n_2 = 5$. This property becomes a basis for computing the cardinality of $X$, which is given by:

In [3]:


lenX = 3 *5**2 *6
display(
    Latex(r"$|X| = n_1 n_2^2 |D_{conc}| =$ " + f"{lenX}")
)

<IPython.core.display.Latex object>

This number is relatively small.

### Empirical Computational Time

We will now evaluate the empirical computational time of the Python script implementation of $\Psi$. Note that this aspect is also dependent to the machine in use when executing the code. Here is the detail of our machine specification:
- CPU AMD Ryzen 5 4500H Hexa-Core 12 Threads
- 16 GB RAM DDR4 Dual Channel

The empirical computational time is presented as follows:

In [4]:
def empirical_time(output= False):
    x = (11, 3, 3, 5, 1, 1, 3, 5, 5)
    start = time.time()
    Psi(x)
    end = time.time()
    emp_time =  round(end - start, 3)
    print(f"Empirical Computational Time: {emp_time} sec")
    if output == True:
        return emp_time

emp_time = empirical_time(output= True)

Empirical Computational Time: 0.231 sec


### Estimated Total Computational Time

In [5]:

display(
    Latex(r"Estimated Total Time $\approx$ " + f"{round(emp_time *lenX / 60, 3)} mins")
)

<IPython.core.display.Latex object>

This a is considerably small amount of time for computation. Thus, it concludes the use of ES method.

# 3. Computation

In [6]:
# Constructing the feasible set:
D_conc = [3, 7, 10, 14, 21, 28]
X = [
    [Ap, As, 3, 5, 1, 1, 3, d4, d4]
    for Ap in range(1, 16)
    for As in range(1, 6)
    for d4 in D_conc
]

def optimES(verbose= True, all_output= False):
    X46_costs = dict()
    comp_time = list()

    rent_pFORM = 469442.31
    rent_pSHOR = 312961.54
    rent_sFORM = 928471.15
    rent_sSHOR = 618980.77

    if verbose == True:
        print("ITERATION TIME")
        print("=" *len("ITERATION TIME"))
    
    for x, k in zip(X, range(0, len(X) + 1)):
        sort_start = time.time()
        cgs = CGSMPDT1(*beta(x))
        frakT = cgs.cgsF()
        if frakT <= 46:
            X46_costs[tuple(x)] = Psi(
                x,
                rent_pFORM= rent_pFORM,
                rent_pSHOR= rent_pSHOR,
                rent_sFORM= rent_sFORM,
                rent_sSHOR= rent_sSHOR
            )
            
        sort_end = time.time()
        if verbose == True:
            sorting_time = sort_end - sort_start
            comp_time.append(sorting_time)
            print(f"Iteration {k} " + "." *60 + f" {round(sorting_time, 2)} sec")

    other_start = time.time()
    min_value = min(X46_costs.values())
    solutions = [x for x, y in X46_costs.items() if y == min_value]
    other_end = time.time()
    other_time = other_end - other_start
    comp_time.append(other_time)

    if verbose == True:
        print(f"\nTOTAL TIME (with other computations): {round(sum(comp_time), 2)} sec")
    
    if all_output == True:
        return solutions, X46_costs
    else:
        return solutions

the_solution = optimES(all_output= True)

ITERATION TIME
Iteration 0 ............................................................ 0.24 sec
Iteration 1 ............................................................ 0.23 sec
Iteration 2 ............................................................ 0.23 sec
Iteration 3 ............................................................ 0.23 sec
Iteration 4 ............................................................ 0.23 sec
Iteration 5 ............................................................ 0.22 sec
Iteration 6 ............................................................ 0.23 sec
Iteration 7 ............................................................ 0.22 sec
Iteration 8 ............................................................ 0.22 sec
Iteration 9 ............................................................ 0.22 sec
Iteration 10 ............................................................ 0.22 sec
Iteration 11 ............................................................ 0.22 sec

# 4. Result

In [7]:
# RESULT:
def frakT(x):
    cgs = CGSMPDT1(*beta(x))
    return cgs.cgsF()

opt_solution = the_solution[0][0]
cost_opt_sol = Psi(opt_solution)
time_opt_sol = frakT(opt_solution)

result_html = f"""
<div style="
    font-size: 18px;
    ">
    Solution: {opt_solution}<br>
    Optimal Cost: IDR {round(cost_opt_sol/10**9, 3)}B<br>
    Optimal Time: {time_opt_sol} days
</div>
"""
display(HTML(result_html))

From the result, we conclude that the optimal configuration is described as follows:
- Number of formwork and shoring for primary area: 4
- Number of formwork and shoring for secondary area: 2
- Age of concrete with immediate strength: 3 days

This configuration gives us a completion concreting time of **38 days** with a total cost of **IDR 1.199B**. This result is even better compared to the manually optimized solution which has a completion concreting time of 46 days with a total cost of IDR 1.265B.